In [10]:
path="/mnt/nfs/zyxing/msu/dance_temp/dance/examples/search/generate_pseudocode/openevolve_output/checkpoints/checkpoint_20/programs/a8e4a0b8-a492-40eb-9ed3-2a6d9513beb6.json"
import json
with open(path,"r") as f:
    data=json.load(f)

In [14]:
print(data['code'])

"""Reimplementation of the scDeepSort cell-type annotation method.

Reference
---------
Shao, Xin, et al. "scDeepSort: a pre-trained cell-type annotation method for single-cell transcriptomics using deep
learning with a weighted graph neural network." Nucleic acids research 49.21 (2021): e122-e122.

"""
import time
from contextlib import nullcontext
from copy import deepcopy
from pathlib import Path

import dgl
import torch
import torch.nn as nn
from dgl.dataloading import DataLoader, NeighborSampler

from dance.models.nn import AdaptiveSAGE
from dance.modules.base import BaseClassificationMethod
from dance.transforms import Compose, SetConfig
from dance.transforms.graph import PCACellFeatureGraph
from dance.typing import LogLevel, Optional


class GNN(nn.Module):
    """The scDeepSort GNN model.

    Message passing between cell and genes to learn cell representations for cell-type annotations. The graph contains
    both cell and gene nodes. The gene features are initialized as PCA e